In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGCHAIN_API_KEY")
LANGCHAIN_PROJECT=os.getenv("LANGCHAIN_PROJECT")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]=LANGCHAIN_PROJECT

In [2]:
LANGCHAIN_PROJECT

'langgraph_prerequisites'

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Users\Admin\Documents\ML_Projects\Agentic_AI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2472.83it/s]


In [7]:
from langchain_groq import ChatGroq
llm=ChatGroq(model_name="llama-3.1-8b-instant")

In [8]:
llm.invoke("hi,how are you?")

AIMessage(content="I'm just a computer program, so I don't have emotions like humans do, but I'm functioning properly and ready to help you with any questions or tasks you may have. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 41, 'total_tokens': 85, 'completion_time': 0.056951148, 'completion_tokens_details': None, 'prompt_time': 0.002774941, 'prompt_tokens_details': None, 'queue_time': 0.051535854, 'total_time': 0.059726089}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dcfa8-744e-7f73-9ab7-57e8e1f084f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 44, 'total_tokens': 85})

### Predefine tools

In [10]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper()
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
print(tool.run({"query":"langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations engines, object detection, and retrieval-augmented generation (RAG).
Vector embeddings are mathematical representations of data in a high-dimensional s

In [11]:
from langchain_community.tools import YouTubeSearchTool
tool = YouTubeSearchTool()
tool.run("Carry Minati")

"['https://www.youtube.com/watch?v=lWBj9z2xDDs&pp=ygUMQ2FycnkgTWluYXRp', 'https://www.youtube.com/watch?v=Qy3OLvNNjZU&pp=ygUMQ2FycnkgTWluYXRp']"

In [12]:
from langchain_community.tools.tavily_search import TavilySearchResults
tool = TavilySearchResults()
tool.invoke({"query": "Top selling cars in India"})

[{'title': 'Maruti Suzuki Dzire Tops India Car Sales In March 2026, Beats Nexon, Creta, Punch',
  'url': 'https://www.ndtv.com/auto/maruti-suzuki-dzire-tops-india-car-sales-in-march-2026-beats-nexon-creta-punch-11313180',
  'content': "Further strengthening Maruti's hold on the top 10 list was the Baleno (7th best-selling car in March 2026), which posted 16,392 units in March 2026. The Brezza, another key SUV from the brand, recorded 16,130 units, while the sporty Fronx continued its momentum with 15,540 units.\n\nHyundai Creta\n\nHyundai Creta\n\nRounding off the list was the Mahindra Scorpio range, including the Scorpio N, with 14,578 units, highlighting the sustained demand for rugged lifestyle SUVs.\n\nThe overall top 10 chart clearly reflects the market's current balance between traditional passenger cars and utility vehicles. While SUVs continue to dominate buyer interest across segments, the Dzire's return to the top spot shows that compact sedans still have a strong and loyal c

### Create a Custom tool

In [19]:
from langchain_core.tools import tool
@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

In [20]:
get_word_length.invoke("abc")

3

In [21]:
get_word_length.invoke("sunny savita")

12

In [22]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [23]:
@tool
def summisation(a: int, b: int) -> int:
    """Adding two numbers."""
    return a + b

In [24]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [25]:
print(summisation.name)
print(summisation.description)
print(summisation.args)

summisation
Adding two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [26]:
multiply.invoke({"a":10,"b":20})

200

In [27]:
summisation.invoke({"a":10,"b":20})

30

### Concept of Agents

In [36]:
from langchain.agents import create_agent
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [50]:
# Create tool
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Create agent
agent = create_agent(
    model=llm,   # your LLM (e.g., ChatOpenAI or Ollama)
    tools=[wiki],
    system_prompt="You are a helpful assistant",
)

# Run
response = agent.invoke({"query": "Who is Elon Musk?"})
print(response)

{'messages': [AIMessage(content="I'm ready to assist you. What would you like to know?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 281, 'total_tokens': 296, 'completion_time': 0.04364307, 'completion_tokens_details': None, 'prompt_time': 0.018878385, 'prompt_tokens_details': None, 'queue_time': 0.051100205, 'total_time': 0.062521455}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd9a2-b738-7aa3-909e-71f6533320b8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 281, 'output_tokens': 15, 'total_tokens': 296})]}


In [44]:
agent.invoke({"query": "what is llama and who create this llm model?"})


{'messages': [AIMessage(content="I'm ready to assist you. What would you like to do?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 281, 'total_tokens': 296, 'completion_time': 0.038625017, 'completion_tokens_details': None, 'prompt_time': 0.016168298, 'prompt_tokens_details': None, 'queue_time': 0.053414351, 'total_time': 0.054793315}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd99e-8bf2-7ae0-9d0a-9fa331be5775-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 281, 'output_tokens': 15, 'total_tokens': 296})]}

In [47]:
from langchain_community.tools.tavily_search import TavilySearchResults
search = TavilySearchResults()
search.invoke("what is the weather in Bangalore now")

[{'title': 'Bangalore Weather Today, Bangalore Temperature and Air Quality (2026-04-29) - India Today',
  'url': 'https://www.indiatoday.in/weather/bangalore-weather-forecast-today',
  'content': "## Weather In Bangalore\n\nThe minimum temperature in Bangalore today is likely to hover around 24 degrees Celsius, while the maximum temperature might reach 37 degrees Celsius. The mercury level is expected to hover around 34 degrees Celsius throughout the day, with the wind speed around 5.54. The wind will move around 22 degrees with a gust speed of 7.14. The sunrise time is 06:00 AM, while it will set at 06:35 PM on Wednesday. As per the seven-day weather prediction, the temperature in Bangalore is likely to reach 37 degrees Celsius on Wednesday, 36 degrees Celsius on Thursday, 37 degrees Celsius on Friday, 37 degrees Celsius on Saturday, 37 degrees Celsius on Sunday, 33 degrees Celsius on Monday and 34 degrees Celsius on Tuesday.\n\n## Weather of Major Cities [...] ### TRENDING TOPICS\n\n

In [66]:
from langchain.agents import create_agent
from langchain_core.tools import tool

agent = create_agent(
    model=llm,
    tools=[wiki],
    system_prompt="You are a helpful assistant"
)

response = agent.invoke({"input": "hello how are you?"})
print(response)

{'messages': [AIMessage(content='What can I help you with today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 281, 'total_tokens': 290, 'completion_time': 0.017002755, 'completion_tokens_details': None, 'prompt_time': 0.018894438, 'prompt_tokens_details': None, 'queue_time': 0.053883471, 'total_time': 0.035897193}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd9ab-6281-7b31-92a5-684dab6106dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 281, 'output_tokens': 9, 'total_tokens': 290})]}


In [67]:
response = agent.invoke({"input": "whats the weather in Bangalore now?"})
print(response)

{'messages': [AIMessage(content='I can assist you with information and perform tasks. What would you like to know or do?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 281, 'total_tokens': 301, 'completion_time': 0.045988319, 'completion_tokens_details': None, 'prompt_time': 0.018103794, 'prompt_tokens_details': None, 'queue_time': 0.051170159, 'total_time': 0.064092113}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd9ab-63fd-7b80-b0c7-5aeae0e7bf10-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 281, 'output_tokens': 20, 'total_tokens': 301})]}


RAG tool

ReAct

custom tool with ReAct agent

agent code from latest versions(v0.2 and v0.3)

### RAG

In [69]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [71]:
loader = WebBaseLoader(
    "https://docs.smith.langchain.com/overview",
    header_template={
        "User-Agent": "Mozilla/5.0"
    }
)
docs = loader.load()

In [72]:
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
).split_documents(docs)

In [73]:
vector = FAISS.from_documents(documents,embeddings)
retriever = vector.as_retriever()

In [74]:

retriever.invoke("how to upload a dataset")[0]

Document(id='cf418cd1-edd3-45c9-a9f5-3115b77f8175', metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='\u200bMore ways to build\nFleetDesign and deploy AI agents visually without writing code.Build an agentPrompt engineeringIterate on prompts with built-in versioning and collaboration to ship improvements faster.Test your promptsLangSmith CLIQuery and manage traces, datasets, experiments, and more from the terminal.Use the CLIStudioUse a visual interface to design, test, and refine applications end-to-end.Develop with Studio\n\u200bInfrastructure\nPlatform setupUse LangSmith in managed cloud, in a self-hosted environment, or hybrid to match your infrastructure and compliance needs.Choose how to set up LangSmithSecurity & complianceLangSmith meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. Meet with our team to learn more or visit our Trust

In [76]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!",
)

In [78]:
tools = [wiki, retriever_tool]

In [80]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant"
)



In [81]:
response = agent.invoke({"input": "hi! what is a langsmith?"})
print(response)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2vwn85jm6', 'function': {'arguments': '{"query":"The history of the universe"}', 'name': 'wikipedia'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 371, 'total_tokens': 389, 'completion_time': 0.042720777, 'completion_tokens_details': None, 'prompt_time': 0.025121062, 'prompt_tokens_details': None, 'queue_time': 0.051044731, 'total_time': 0.067841839}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd9b2-c6eb-7db0-accf-6c1d5bdee6b8-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'The history of the universe'}, 'id': '2vwn85jm6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 371, 'output_tokens': 18, 'total_tokens': 389}), ToolMessage(content='Page: Age of the universe\

<built-in method keys of dict object at 0x00000213FE9E1B00>


### Add memory component

In [83]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [84]:
message_history=ChatMessageHistory()

In [85]:
agent_with_chat_history = RunnableWithMessageHistory(
    agent,
    # This is needed because in most real world scenarios, a session id is needed
    # It isn't really used here because we are using a simple in memory ChatMessageHistory
    lambda session_id: message_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

In [88]:
agent_with_chat_history.invoke(
    {"input": "hi! my name is sunny how are you?"},
    # This is needed because in most real world scenarios, a session id is needed
    # It isn't really used here because we are using a simple in memory ChatMessageHistory
    config={"configurable": {"session_id": "firstchat"}},
)

{'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'h007vv7cg', 'function': {'arguments': '{"query":"Genghis Khan"}', 'name': 'wikipedia'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 371, 'total_tokens': 388, 'completion_time': 0.035241179, 'completion_tokens_details': None, 'prompt_time': 0.022369388, 'prompt_tokens_details': None, 'queue_time': 0.051100841, 'total_time': 0.057610567}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd9b8-3915-7421-ac22-91b0e5ce7311-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Genghis Khan'}, 'id': 'h007vv7cg', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 371, 'output_tokens': 17, 'total_tokens': 388}),
  ToolMessage(content='Page: Genghis Khan\nSummary: Genghis Khan (born Temüji

####   ReAct Agent (Reasoning + Acting):
####   Definition:

The ReAct framework combines reasoning and acting in a single loop to handle tasks. The agent uses natural language reasoning (thinking through steps) and task actions (performing tasks like calculations or data retrieval).

The agent uses a combination of reasoning steps to guide actions in real-time, using feedback from those actions to further inform the next step in reasoning.

####   How it works:

Step 1: The agent receives a question or task.
Step 2: It reasons aloud (in natural language) about how to solve it.
Step 3: Based on its reasoning, it takes actions (e.g., searching a database, calculating something).
Step 4: The results of these actions are integrated into its reasoning and may trigger further actions.
Step 5: It repeats the process until it arrives at a solution.

####  Key points:

Combines thinking and doing (reasoning and actions).
Performs iterative steps, updating its process based on action results.
Typically handles complex decision-making scenarios.

####   Example:

Original task: “Calculate the total number of apples in a basket if there are 4 baskets and 7 apples in each.”
Reasoning: "I need to multiply 4 by 7 to get the total number of apples."
Action: Perform the multiplication.
Result: "There are 28 apples."
Reasoning: "I am done."

In [95]:
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain_core.tools import Tool

In [96]:
google_search = GoogleSerperAPIWrapper
tools = [
    Tool(
        name="Intermediate Answer",
        func=google_search.run,
        description="useful for when you need to ask with search",
    )
]

In [97]:
template = '''Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
Begin!
Question: {input}
Thought:{agent_scratchpad}'''

In [99]:
prompt = PromptTemplate.from_template(template)

In [104]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt.template
)

In [ ]:
response = agent.invoke({"input": "Where is the hometown of the 2007 US PGA championship winner and his score?"})
print(response)

### ReAct Agent with Custom tools

In [108]:
from langchain_core.tools import tool
from langchain.agents import create_agent

In [109]:
# Custom tool for the Agent 
@tool
def get_employee_id(name):
  """
  To get employee id, it takes employee name as arguments
  name(str): Name of the employee
  """
  fake_employees = {
    "Alice": "E001",
    "Bob": "E002",
    "Charlie": "E003",
    "Diana": "E004",
    "Evan": "E005",
    "Fiona": "E006",
    "George": "E007",
    "Hannah": "E008",
    "Ian": "E009",
    "Jasmine": "E010"}
  
  return fake_employees.get(name,"Employee not found")

In [110]:
# Custom tool for the Agent 
@tool
def get_employee_salary(employee_id):
  """
  To get the salary of an employee, it takes employee_id as input and return salary
  """
  employee_salaries = {
    "E001": 56000,
    "E002": 47000,
    "E003": 52000,
    "E004": 61000,
    "E005": 45000,
    "E006": 58000,
    "E007": 49000,
    "E008": 53000,
    "E009": 50000,
    "E010": 55000
    }
  return employee_salaries.get(employee_id,"Employee not found")

In [111]:
prompt= "Answer the following questions as best you can. You have access to the following tools:"

In [112]:
tools = [get_employee_salary, get_employee_id]

In [116]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt
)

In [117]:
agent.invoke({"input":"What is the Salary of Evan?"})

{'messages': [AIMessage(content="I'm ready to help. What are the questions?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 325, 'total_tokens': 337, 'completion_time': 0.026416514, 'completion_tokens_details': None, 'prompt_time': 0.02842919, 'prompt_tokens_details': None, 'queue_time': 0.061344389, 'total_time': 0.054845704}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd9c6-5a1e-78b3-ad09-0ec2278b0429-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 325, 'output_tokens': 12, 'total_tokens': 337})]}

###  Self-Ask with Search Agent:
####  Definition:

This approach allows the AI to ask itself follow-up questions when it doesn't immediately know the answer. It performs a recursive questioning process to break down complex queries into smaller, more manageable ones.
Once the AI generates these follow-up questions, it performs an external search (e.g., through Google or an internal database) to gather the necessary information before formulating a response.

####  How it works:

Step 1: Receive a complex question.
Step 2: The agent identifies sub-questions or follow-up questions.
Step 3: It performs a search or fetches answers to these questions from external resources (like a web search).
Step 4: The answers are aggregated to provide a complete response.

####  Key points:

Emphasizes question decomposition.
Relies on external search for sub-questions.
Useful for answering open-ended or broad questions where the answer is not immediately available.

####  Example:

Original question: “How many moons does Jupiter have?”
Sub-question: “What is Jupiter?” (search)
Sub-question: “What are moons?” (search)
Finally, it retrieves the answer: "Jupiter has 79 moons."